# Unitree G1: GPU RL pipeline (MJX + custom PPO)

Settings: **Accelerator = GPU T4 x2**, **Internet = On**, **Persistence = Files**.

Every session: run sections 1–2, then whichever phase cells you need. Cells marked ⏱ <5 min are interactive checks; cells marked 🌙 are long jobs to run with **Save Version → Save & Run All**.

## 1. Get / refresh the code
Copies the attached dataset over `/kaggle/working/g1_rl` every session (keeps `outputs/`). After uploading a new dataset version, click the ⟳ *check for updates* icon next to the dataset in the Input panel first.

In [ ]:
import os, shutil, glob, subprocess
GIT_URL = ""   # optional: "https://github.com/<you>/g1_rl.git" instead of a dataset
WORK = "/kaggle/working/g1_rl"
if GIT_URL:
    if os.path.exists(f"{WORK}/.git"):
        subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", "--depth", "1", GIT_URL, WORK], check=True)
else:
    hits = [p for p in glob.glob("/kaggle/input/**/pyproject.toml", recursive=True) if "g1_rl" in open(p).read()]
    assert hits, "Attach the g1_rl dataset or set GIT_URL"
    shutil.copytree(os.path.dirname(hits[0]), WORK, dirs_exist_ok=True)
os.chdir(WORK)
print(os.getcwd(), sorted(os.listdir()))

## 2. Install GPU JAX + MuJoCo/MJX ⏱ ~2 min

In [ ]:
!bash scripts/install_kaggle.sh

---
## Phase 0: setup check & benchmark (done)

In [ ]:
# ⏱ ~2 min
# !python scripts/check_setup.py --platform gpu

In [ ]:
# 🌙 ~15 min
# !python scripts/benchmark.py --platform gpu --scaling --envs 1024,2048,4096,8192 --out outputs/benchmark.json

---
## Phase 1: environment
### Smoke test ⏱ ~3–4 min
512 robots with random actions, 60-step episodes and fast command changes. It prints PASS/FAIL checks and saves `outputs/phase1_smoke/` (report.json, trajectory, video).

In [ ]:
!python scripts/env_smoke.py --platform gpu

In [ ]:
from IPython.display import Video
Video("outputs/phase1_smoke/traj_env0.mp4", embed=True, width=480)

### Environment unit tests ⏱ ~4 min

In [ ]:
!pip install -q pytest && G1RL_TEST_PLATFORM=gpu python -m pytest -q tests/test_env.py

### Full test suite 🌙 ~8 min

In [ ]:
# !G1RL_TEST_PLATFORM=gpu python -m pytest -q

---
## Next phases
Rewards (2), domain randomization (3), PPO training (4), evaluation (5) and export (6) cells are added as each phase lands.